# Sentiment Classifier Training — SemEval 2017 / tweet_eval

Fine-tunes **DistilBERT** (Sanh et al., 2019) on the **SemEval 2017 Tweet Sentiment** corpus  
(accessed via `cardiffnlp/tweet_eval` on HuggingFace, N=45,615 tweets).  

### Steps before running
1. `Runtime → Change runtime type → T4 GPU` → Save
2. Run **Cell 1 only** first (installs packages + restarts kernel automatically)
3. After restart, run **all remaining cells** from Cell 2 onward

In [ ]:
# ── CELL 1 — Install packages  (run this cell first, alone) ───────────────────
# Pins datasets to 3.x to avoid a torchvision VideoReader conflict in Colab 4.x
!pip install "datasets==3.2.0" "transformers>=4.40.0" accelerate scikit-learn -q
print('Packages installed.')
print('Restarting kernel automatically...')

# Restart the kernel so the pinned versions are active
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── CELL 2 — Check GPU  (run after kernel restarts) ───────────────────────────
import torch
print('GPU available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device        :', torch.cuda.get_device_name(0))
    print('Memory (GB)   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')

import datasets, transformers
print('datasets version    :', datasets.__version__)
print('transformers version:', transformers.__version__)

In [ ]:
# ── CELL 3 — Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_PATH = '/content/drive/MyDrive/sentiment_classifier'
os.makedirs(SAVE_PATH, exist_ok=True)
print('Model will be saved to:', SAVE_PATH)

In [ ]:
# ── CELL 4 — Load SemEval 2017 Tweet Sentiment dataset ────────────────────────
from datasets import load_dataset
from collections import Counter

dataset = load_dataset('cardiffnlp/tweet_eval', 'sentiment')

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}

print(f'Train : {len(dataset["train"]):,} samples')
print(f'Val   : {len(dataset["validation"]):,} samples')
print(f'Test  : {len(dataset["test"]):,} samples')
print()
counts = Counter(dataset['train']['label'])
print('Label distribution (train):')
for lid, name in ID2LABEL.items():
    pct = counts[lid] / len(dataset['train']) * 100
    print(f'  {name:10s}: {counts[lid]:,}  ({pct:.1f}%)')
print()
print('Sample:', dataset['train'][0])

In [ ]:
# ── CELL 5 — Tokenise ─────────────────────────────────────────────────────────
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )

tokenised = dataset.map(tokenize, batched=True, batch_size=512)
tokenised = tokenised.rename_column('label', 'labels')
tokenised.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print('Tokenisation complete.')
print('Input shape (first sample):', tokenised['train'][0]['input_ids'].shape)

In [ ]:
# ── CELL 6 — Load DistilBERT with 3-class classification head ─────────────────
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

In [ ]:
# ── CELL 7 — Train  (~8-12 min on T4 GPU) ────────────────────────────────────
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per_class = f1_score(labels, preds, average=None, labels=[0, 1, 2])
    return {
        'accuracy'    : round(accuracy_score(labels, preds), 4),
        'f1_macro'    : round(f1_score(labels, preds, average='macro'), 4),
        'f1_negative' : round(float(f1_per_class[0]), 4),
        'f1_neutral'  : round(float(f1_per_class[1]), 4),
        'f1_positive' : round(float(f1_per_class[2]), 4),
    }

# warmup_steps = 10% of total steps
# total_steps  = (45615 / 64) * 3 epochs ≈ 2138
WARMUP_STEPS = 200

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=50,
    report_to='none',
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenised['train'],
    eval_dataset=tokenised['validation'],
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()

In [ ]:
# ── CELL 8 — Evaluate on held-out test set ────────────────────────────────────
test_results = trainer.evaluate(tokenised['test'])

print('=' * 55)
print('  TEST SET RESULTS  —  cite these in your thesis')
print('=' * 55)
print(f"  Accuracy    : {test_results.get('eval_accuracy', 0):.4f}")
print(f"  F1 Macro    : {test_results.get('eval_f1_macro', 0):.4f}")
print(f"  F1 Negative : {test_results.get('eval_f1_negative', 0):.4f}")
print(f"  F1 Neutral  : {test_results.get('eval_f1_neutral', 0):.4f}")
print(f"  F1 Positive : {test_results.get('eval_f1_positive', 0):.4f}")
print('=' * 55)

In [ ]:
# ── CELL 9 — Save model + tokeniser to Google Drive ───────────────────────────
import json

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

metrics = {
    'base_model'   : 'distilbert-base-uncased',
    'dataset'      : 'cardiffnlp/tweet_eval (sentiment, SemEval 2017)',
    'train_size'   : len(dataset['train']),
    'val_size'     : len(dataset['validation']),
    'test_size'    : len(dataset['test']),
    'epochs'       : 3,
    'learning_rate': 2e-5,
    'warmup_steps' : WARMUP_STEPS,
    'test_results' : {k.replace('eval_', ''): v for k, v in test_results.items()},
}
with open(f'{SAVE_PATH}/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to Google Drive:', SAVE_PATH)
print('Files:')
for fname in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(f'{SAVE_PATH}/{fname}')
    print(f'  {fname:40s}  {size/1e6:.1f} MB')

In [ ]:
# ── CELL 10 — Download model as ZIP ──────────────────────────────────────────
import shutil
from google.colab import files

zip_path = '/content/sentiment_classifier'
shutil.make_archive(zip_path, 'zip', SAVE_PATH)

zip_size = os.path.getsize(f'{zip_path}.zip') / 1e6
print(f'ZIP size: {zip_size:.0f} MB')
print('Starting download...')
files.download(f'{zip_path}.zip')

## After downloading

1. Unzip `sentiment_classifier.zip`
2. Place the folder at:
   ```
   m3_implementation/memory/models/sentiment_classifier/
   ```
3. System auto-detects and uses your trained model on next startup — no other changes needed.

`training_metrics.json` inside the folder has the test results to cite in your thesis.

---
## Evaluation Analysis (run independently — no retraining needed)
The cells below load your **already-trained model** from Google Drive and generate full evaluation visualisations.  
Run from Cell E1 onward — no need to re-run training cells.

In [ ]:
# ── CELL E1 — Setup: mount Drive + install packages ───────────────────────────
!pip install "datasets==3.2.0" "transformers>=4.40.0" scikit-learn seaborn -q

from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_PATH = '/content/drive/MyDrive/sentiment_classifier'
EVAL_PATH  = '/content/drive/MyDrive/sentiment_classifier/eval_results'
os.makedirs(EVAL_PATH, exist_ok=True)

print('Model path :', MODEL_PATH)
print('Eval output:', EVAL_PATH)
# Verify model files exist
for f in ['config.json', 'model.safetensors', 'tokenizer.json']:
    status = '✓' if os.path.isfile(f'{MODEL_PATH}/{f}') else '✗ MISSING'
    print(f'  {f}: {status}')

In [ ]:
# ── CELL E2 — Load trained model + test set, run predictions ──────────────────
import numpy as np
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from datasets import load_dataset

ID2LABEL = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
LABEL2ID = {'Negative': 0, 'Neutral': 1, 'Positive': 2}

# Load trained model
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
model     = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'Model loaded — running on {device}')

# Load test set
dataset   = load_dataset('cardiffnlp/tweet_eval', 'sentiment')
test_data = dataset['test']
print(f'Test set: {len(test_data):,} samples')

# Batch inference
BATCH = 128
all_preds, all_labels = [], []

for i in range(0, len(test_data), BATCH):
    batch_texts  = test_data['text'][i:i+BATCH]
    batch_labels = test_data['label'][i:i+BATCH]
    enc = tokenizer(batch_texts, truncation=True, max_length=128,
                    padding='max_length', return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    preds = torch.argmax(logits, dim=-1).cpu().numpy()
    all_preds.extend(preds)
    all_labels.extend(batch_labels)

y_pred = np.array(all_preds)
y_true = np.array(all_labels)
print(f'Inference complete — {len(y_pred):,} predictions')

In [ ]:
# ── CELL E3 — VISUALISATION 1: Confusion Matrix ───────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

labels = ['Negative', 'Neutral', 'Positive']
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrix — DistilBERT on SemEval 2017 Test Set (12,284 samples)',
             fontsize=13, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0],
            linewidths=0.5, linecolor='gray', annot_kws={'size': 13})
axes[0].set_title('Raw Counts', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('True', fontsize=11)

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1],
            linewidths=0.5, linecolor='gray', annot_kws={'size': 13})
axes[1].set_title('Row-normalised (%)', fontsize=12)
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('True', fontsize=11)

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# ── CELL E4 — VISUALISATION 2: Per-class Precision / Recall / F1 ──────────────
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)

metrics_list = ['precision', 'recall', 'f1-score']
x      = np.arange(len(labels))
width  = 0.25
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(10, 6))
for i, (metric, color) in enumerate(zip(metrics_list, colors)):
    values = [report[label][metric] for label in labels]
    bars = ax.bar(x + i * width, values, width, label=metric.capitalize(),
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

macro_f1 = report['macro avg']['f1-score']
ax.axhline(y=macro_f1, color='black', linestyle='--', linewidth=1.5,
           label=f'Macro F1 = {macro_f1:.4f}')

ax.set_xlabel('Sentiment Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-class Metrics — DistilBERT on SemEval 2017 Test Set', fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 1.10)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFull Classification Report:')
print(classification_report(y_true, y_pred, target_names=labels))
print('Saved: per_class_metrics.png')

In [ ]:
# ── CELL E5 — VISUALISATION 3: Metrics Summary Card ──────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import accuracy_score, f1_score
import json

acc      = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_neg   = f1_score(y_true, y_pred, average=None)[0]
f1_neu   = f1_score(y_true, y_pred, average=None)[1]
f1_pos   = f1_score(y_true, y_pred, average=None)[2]

fig, ax = plt.subplots(figsize=(8, 5))
ax.axis('off')
fig.patch.set_facecolor('#f8f9fa')

title = 'DistilBERT Sentiment Classifier — Final Evaluation'
ax.text(0.5, 0.95, title, ha='center', va='top', fontsize=14,
        fontweight='bold', transform=ax.transAxes)
ax.text(0.5, 0.87, 'Dataset: SemEval 2017 Tweet Sentiment  |  Test set: 12,284 samples',
        ha='center', va='top', fontsize=10, color='gray', transform=ax.transAxes)

rows = [
    ('Accuracy',        f'{acc:.4f}',     f'{acc*100:.2f}%'),
    ('Macro F1',        f'{f1_macro:.4f}',f'{f1_macro*100:.2f}%'),
    ('F1 — Negative',   f'{f1_neg:.4f}',  ''),
    ('F1 — Neutral',    f'{f1_neu:.4f}',  ''),
    ('F1 — Positive',   f'{f1_pos:.4f}',  ''),
]

col_labels = ['Metric', 'Score', '']
table_data = [[r[0], r[1], r[2]] for r in rows]

table = ax.table(cellText=table_data, colLabels=col_labels,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.0)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#343a40')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#e9ecef')
    cell.set_edgecolor('white')

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Save JSON summary
eval_summary = {
    'model': 'distilbert-base-uncased (fine-tuned)',
    'dataset': 'cardiffnlp/tweet_eval sentiment (SemEval 2017)',
    'test_samples': int(len(y_true)),
    'accuracy': round(float(acc), 4),
    'f1_macro': round(float(f1_macro), 4),
    'f1_negative': round(float(f1_neg), 4),
    'f1_neutral': round(float(f1_neu), 4),
    'f1_positive': round(float(f1_pos), 4),
}
with open(f'{EVAL_PATH}/eval_summary.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print('Saved: metrics_summary.png')
print('Saved: eval_summary.json')
print('\nEvaluation files in:', EVAL_PATH)